# <center >谷歌开源Agent框架ADK开发实战</center>
## <center>Part 11. Google ADK 集成 MCP 服务及认证组件开发实战</center>

&emsp;&emsp;本节课程我们主要围绕`Google ADK`框架如何接入非常热门的`MCP`服务展开详细的介绍。此外，除了较为基础的`ADK + MCP`集成实战内容外，我们还将重点介绍在`ADK`框架下使用`MCP`的一些优化技巧，以及如何巧妙的应用`Google ADK`内置的身份认证组件来实现对`MCP`服务的层级权限控制。

&emsp;&emsp;从`Google ADK`的应用生态来看，该框架目前完全遵循 `Model Context Protocol (MCP)` 标准，支持`Stdio`、`HTTP+SSE`传输及`Streamable HTTP`三种传输协议的接入支持。如下表所示：

<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>ADK 支持的 MCP 通信方式及对应实现</font></p>
<div class="center">


| **通信方式**            | **MCP 标准描述**                                                                    | **ADK 实现与配置**                                                                                                   |
| ------------------- | ------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------- |
| **Stdio** | 客户端以子进程方式启动`MCP`服务器，通过 `stdin/stdout` 交换 `JSON-RPC` 消息。                               | ADK 使用 `StdioServerParameters`，在 `MCPToolset` 中指定要运行的命令和参数，启动本地`MCP`服务器并通过标准IO通信。                                 |
| **HTTP + SSE**      | 服务器独立运行，客户端通过`HTTP POST`发送请求，通过 `SSE (HTTP GET)` 建立长连接接收服务器推送的消息（需要两个端点）。           | ADK 使用 `SseServerParams`，提供远程 `SSE` 端点的 URL 及认证头等配置，实现与远程 `MCP` 服务的双通道通信。                                           |
| **Streamable HTTP** | 单一`HTTP`端点同时处理请求和响应的新协议（2025年推出），服务器通过一个端点即可接受请求并以流式方式返回多条消息，可选地使用 `SSE` 格式发送多部分响应。 | ADK 使用 `StreamableHTTPConnectionParams`（Python ADK）来连接支持 `Streamable HTTP` 的 `MCP` 服务器，以单端口方式进行交互（此为远程`MCP`的新推荐传输方式）。 |

&emsp;&emsp;在接入形式上，`ADK` 提供了统一的机制将 `MCP` 工具接入到`Agent`实例中，核心组件就是 `MCPToolset` 类。其核心源码定义位置：https://github.com/google/adk-python/blob/main/src/google/adk/tools/mcp_tool/mcp_toolset.py

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507251349432.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;`MCPToolset` 是 `ADK` 中用于与 `MCP` 服务器交互的核心类。该类中实现的核心功能就是将`MCP`实例加入`Agents`的 `tools` 列表后，便可连接 `MCP` 服务器、发现其工具并将这些工具适配为 `ADK` 工具。同时，根据不同的`MCP`通信协议，利用 `MCPToolset` 可以通过传递不同的连接信息`connection_params` 来实现不同的通信方式。如下图所示：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507301411760.png" alt="MCPToolset" style="width: 80%;">
</div>


&emsp;&emsp;无论选择哪种连接方式，都可以通过统一的内置功能来实现与`MCP`服务器的交互，包含如下几个方面：

<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>MCPToolset 功能</font></p>
<div class="center">

| 特征 (Feature)         | 描述                                                                                   |
|------------------------|----------------------------------------------------------------------------------------|
| 连接管理 (`Connection Management`)  | 支持 `stdio`、`SSE` 和 `Streamable HTTP` 方式连接 `MCP` 服务器                                             |
| 工具发现 (`Tool Discovery`)         | 自动发现 `MCP` 工具并包装为 `ADK` 工具                                                     |
| 工具过滤 (`Tool Filtering`)         | 支持按名称或自定义条件过滤工具                                                         |
| 身份验证 (`Authentication`)         | 支持多种工具调用的身份验证方案                                                         |
| 资源清理 (`Resource Cleanup`)       | 自动管理连接的清理和资源释放                                                           |

- **连接管理**：`MCPToolset` 在初始化时建立与 `MCP` 服务器的连接，支持本地进程通信（`StdioServerParameters`）或远程 `SSE/HTTP` 连接（`SseServerParams`、`StreamableHTTPConnectionParams`），并在代理退出时会优雅地关闭连接；
- **工具发现与适配**：`MCPToolset` 调用 `list_tools` 方法从服务器获取工具列表，并将 `MCP` 工具模式转换为 `ADK` 的 `BaseTool` 实例，适配后的工具在代理中表现为本地工具，代理可直接调用，当代理调用这些工具时，`MCPToolset` 会将请求转发给 `MCP` 服务器并返回结果；
- **工具过滤**：可通过 `tool_filter` 指定要暴露的工具子集，避免将所有远程工具暴露给代理。

&emsp;&emsp;如下所示，在源码中实现的方法中，通过`get_tools`方法获取`MCP`工具列表，并将其转换为`ADK`的`BaseTool`实例，并通过`close()`方法在调用结束后执行清理并释放`MCP`会话资源。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507251423525.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;此外还需要了解的是，`MCPTool` 类包装了单独的 `MCP` 工具并扩展了 `BaseAuthenticatedTool` 以提供身份验证功能。其核心定义如下：

```python
    # adk-python-main\src\google\adk\tools\mcp_tool\mcp_toolset.py
      async def get_tools(
      self,
      readonly_context: Optional[ReadonlyContext] = None,
    ) -> List[BaseTool]:
        .....
        .....
        # Apply filtering based on context and tool_filter
        tools = []
        for tool in tools_response.tools:
        mcp_tool = MCPTool(
            mcp_tool=tool,
            mcp_session_manager=self._mcp_session_manager,
            auth_scheme=self._auth_scheme,
            auth_credential=self._auth_credential,
        )

        if self._is_tool_selected(mcp_tool, readonly_context):
            tools.append(mcp_tool)
        return tools
```

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507301425349.png" alt="MCPTool" style="width: 80%;">
</div>


&emsp;&emsp;将`MCP`工具转化成`ADK`支持的工具类型的转化过程依次包括：先从 `MCP JSON` 模式到 `ADK` 内置函数声明的模式转换，再从 `ADK` 内置函数声明的模式到 `ADK` 的 `BaseTool` 实例，同时支持身份验证标头提取和管理，以及自动感知工具执行，在连接失败时会自动重试。

&emsp;&emsp;另一方面，`MCPSessionManager` 通过复杂的连接管理和池功能处理 `MCP` 客户端会话。其核心源码实现如下：

```python
    class MCPToolset(BaseToolset):
        def __init__(
            self,
            *,
            ....
        ):
            super().__init__(tool_filter=tool_filter)

            ....
            ....

            # Create the session manager that will handle the MCP connection
            self._mcp_session_manager = MCPSessionManager(
                connection_params=self._connection_params,
                errlog=self._errlog,
            )
            self._auth_scheme = auth_scheme
            self._auth_credential = auth_credential

        @retry_on_closed_resource
        async def get_tools(
            self,
            readonly_context: Optional[ReadonlyContext] = None,
        ) -> List[BaseTool]:

            ....
            ....

            # Apply filtering based on context and tool_filter
            tools = []
            for tool in tools_response.tools:
            mcp_tool = MCPTool(
                mcp_tool=tool,
                mcp_session_manager=self._mcp_session_manager,
                auth_scheme=self._auth_scheme,
                auth_credential=self._auth_credential,
            )

            if self._is_tool_selected(mcp_tool, readonly_context):
                tools.append(mcp_tool)
            return tools

        async def close(self) -> None:

            try:
                await self._mcp_session_manager.close()
            except Exception as e:
            # Log the error but don't re-raise to avoid blocking shutdown
                print(f"Warning: Error during MCPToolset cleanup: {e}", file=self._errlog)
```

- 会话池 ：根据身份验证标头对会话进行池化，以实现高效重用；
- 自动重新连接 ：使用 `@retry_on_closed_resource` 装饰器来处理断开的会话；
- 连接验证 ：在重用之前检查会话健康状况，并在需要时创建新会话。

&emsp;&emsp;以上就是`Google ADK`与外部`MCP`服务进行交互时在会话管理、工具发现与适配、身份验证、资源清理等方面的实现原理。在这种模式下，我们需要重点了解的是，`ADK` 框架与 `MCP` 实现的是双向的交互模式，即`ADK` 可以作为 `MCP` 的客户端，也可以作为 `MCP` 的服务端。即：

- **作为 `MCP` 客户端使用现有服务器**：`ADK` 代理通过 `MCPToolset` 访问外部 `MCP` 服务器提供的工具。这是最常见的方式，适合需要利用第三方能力的代理。例如，利用文件系统、谷歌地图、GitHub 等 `MCP` 服务器提供的功能。

- **将 `ADK` 工具暴露为 `MCP` 服务器**：开发者可用 `ADK` + `mcp/fastmcp` 库创建 `MCP` 服务器，将 `ADK` 工具包装并公开给其他 `MCP` 客户端。这种模式用于将自定义工具分享给更广泛的生态。

&emsp;&emsp;本节课，我们主要围绕`ADK`作为客户端接入外部`MCP`服务器的场景展开介绍。而如何将自身暴露为`MCP`服务，我们将在下一节课中结合更具体的案例展开说明。

# 2. 使用外部 MCP 服务器

&emsp;&emsp;`Google ADK`框架是从 `v1.2`版本起便可以通过`MCPToolset` 集成于直接连接 `MCP` 服务器（现在的最新版本是`v1.8.0`）。在这种模式下， `ADK Agent` 可以作为`MCP`客户端，从外部 `MCP` 服务器获取工具并使用。我们在进行开发时，开只需在`Agent`定义时将一个 `MCPToolset` 实例加入工具列表，框架便会自动处理与指定 `MCP` 服务器的连接、工具发现和调用代理等工作。而目前主流`MCP`的使用形式可以归纳为两大类：<font color="red">**自托管启动与平台托管直连。**</font>

- **自托管启动（Self-hosted）**

&emsp;&emsp;这种方式主要是应用[`MCP`官方的Python / TypeScript MCP SDK](https://github.com/modelcontextprotocol) 把 `Server` 代码跑起来（FastAPI / Express 集成）, 程序启动后会打印或返回一个本地或内网 `SSE URL`，然后便可以用这个 `URL` 建立事件流接入到`OpenAI Agents SDK` 框架中。如果需要使用`MCP` 官方的`Python` SDK构建`SSE` 模式的`MCP` 服务器，可以学习《【加餐】MCP SSE与流式HTTP开发实战】》课程模块，我们这里不再重复讲解。

- **平台托管直连（Platform-hosted）**

&emsp;&emsp;诸如 [`ModelScope`的 `MCP` 广场](https://www.modelscope.cn/mcp)、[mcp.run](https://docs.mcp.run/integrating/tutorials/mcp-run-sse-openai-agents)等社区维护大量 `MPC` 服务并支持一键启动`MCP`的`SSE` 模式。使用的方法非常简单，只需要在平台首页选好`MCP Server` → 生成一个`API-Key`，一键生成 `SSE URL` ，即可直接接入使用。

&emsp;&emsp;对于国内开发者，通过“一键复制 URL + API-Key”就能拿到 `SSE` 端点的热门网站大致可以分为四类：
1. 专门的 MCP 服务市场/目录（如 MCP Market、AIbase、MCP Server Hub）；
2. 云厂商推出的 官方托管平台（阿里云 “百炼” 与 Higress Marketplace）；
3. 行业 API 已内置 MCP & SSE（百度地图、腾讯位置服务等）；
4. IDE / 客户端集市 把第三方 SSE URL 聚合到本地（Cherry Studio、Cursor）。

<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>MCP 热门导航网站</font></p>
<div class="center">

| 平台                            | 规模与特点                                                                      | 访问链接            |
| ----------------------------- | -------------------------------------------------------------------------- | ---------------- |
| **ModelScope MCP 广场** | 超过 3000+ 个中文托管 MCP；详情页右侧直接复制 SSE URL，部分服务需在“API Key”栏目填 Token | [点击进入](https://www.modelscope.cn/mcp/) |
| **MCP Market** | 标榜“国内首个 MCP 服务市场”，已收录 **1W+** SSE Server，并提供云托管与 Playground，一键生成专属 URL | [点击进入](https://mcpmarket.cn/) |
| **AIbase MCP 资源站**            | 整理 GitHub 上的热门 MCP Server 并直接给出 SSE 地址与使用教程                  | [点击进入](https://mcp.aibase.cn/)   |
| **MCP Server Hub**            | 国际化，标签筛选 + “Copy URL” 操作流畅，站内统计数千条 SSE Server                    | [点击进入](https://mcpserverhub.com/)       | |
| **阿里云 百炼**                  | 官方托管“全周期 MCP 服务”，控制台能生成已鉴权的 SSE URL              | [点击进入](https://bailian.console.aliyun.com/?spm=5176.29619931.0.0.1369521cW5hVGj&tab=mcp#/mcp-market) |


&emsp;&emsp;接下来我们就针对这三种接入形式，分别进行详细的介绍。

## 2.1 Stdio 通信模式

&emsp;&emsp;按照`MCP` 官方规范，`stdio` 传输模式指的是客户端用子进程的方式启动 `MCP` 服务器，双方通过该子进程的 `STDIN/STDOUT` 管道直接交换 `JSON‑RPC` 消息。因此首先需要明确的是：`Stdio`传输模式的`MCP`需要`MCP Server`和`MCP Client` 运行在同一台机器、同一个 `OS` 进程树内，天然无法通过局域网甚至本机 `127.0.0.1` 以外的 `TCP` 套接字访问。而要在不同主机之间通信，应改用 `Streamable‑HTTP / SSE` 之类的网络传输。

&emsp;&emsp;`Google ADK` 是在`MCPSessionManager`中通过统一管理不同类型的连接参数，来实现对不同类型的 `MCP` 服务器的支持。其中`StdioConnectionParams` 是用于 `stdio` 类型的 `MCP` 服务器，其核心处理架构源码如下所示：

```python

    # adk-python-main\src\google\adk\tools\mcp_tool\mcp_session_manager.py
    if isinstance(self._connection_params, StdioConnectionParams):
      client = stdio_client(
          server=self._connection_params.server_params,
          errlog=self._errlog,
      )
```

&emsp;&emsp;`StdioConnectionParams` 是`MCPSessionManager`的属性，用于存储`stdio`类型的`MCP`服务器连接参数。其核心定义如下：

```python

    # adk-python-main\src\google\adk\tools\mcp_tool\mcp_session_manager.py
    class StdioConnectionParams(BaseModel):
    """Parameters for the MCP Stdio connection.

    Attributes:
        server_params: Parameters for the MCP Stdio server.
        timeout: Timeout in seconds for establishing the connection to the MCP
            stdio server.
    """

        server_params: StdioServerParameters
        timeout: float = 5.0
```

&emsp;&emsp;`StdioConnectionParams` 一共只有两个参数属性，分别是`server_params`和`timeout`。其中`server_params`是`StdioServerParameters`类型的对象，用于存储`stdio`类型的`MCP`服务器连接参数。`StdioServerParameters` 参数用于告诉运行时如何在本机以 `stdio` 传输方式启动并管理 `MCP` 服务器进程，因此它承担了命令行、环境变量、工作目录及字符编解码等所有本地进程层面的配置职责。如下所示：


<style>
.center {
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size="4">StdioServerParameters 参数</font></p>
<div class="center">

| 参数 | 类型 / 默认值 | 说明 | 常见用法示例 |
| --- | --- | --- | --- |
| `command` | `str` **(必填)** | 要执行的可执行文件或脚本的绝对/相对路径。代理通过 `subprocess` 把它作为子进程拉起。 | `python`、`mcp-flight-search`、`./server` |
| `args` | `list[str]`，默认 `[]` | 启动命令行参数。常用于指定 `--connection_type stdio`、模型路径等。 | `["--connection_type", "stdio"]` |
| `env` | `dict[str, str] \| None` | 启动子进程时注入/覆盖的环境变量；若为 `None` 则继承当前 shell 环境。 | `{"SERP_API_KEY": os.getenv("SERP_API_KEY")}` |
| `cwd` | `str \| Path \| None` | 子进程工作目录；默认继承父进程。可解决相对路径依赖或找不到文件的问题。 | `Path(__file__).parents[1] / "servers/weather"` |
| `encoding` | `str = "utf-8"` | 与 MCP 进程交换 JSON 消息时使用的字符编码。 | `"utf-8"`、`"gbk"` |
| `encoding_error_handler` | `Literal["strict", "ignore", "replace"] = "strict"` | 字符集解码错误策略：`strict` 抛异常；`ignore` 跳过非法字节；`replace` 用 ？替换。 | `replace`（日志含混合编码时常用） |
</div>


&emsp;&emsp;因此，`Google ADK`框架中接入`stdio`类型`MCP`服务器时的伪代码如下所示：

```python
    from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
    from google.adk.tools.mcp_tool.mcp_toolset import StdioServerParameters, StdioConnectionParams  

    # 定义 MCP 服务器
    amap_mcp_server = MCPToolset(
        connection_params=StdioConnectionParams(
            server_params=StdioServerParameters(
                command="python",
                args=[server_path],
                env={"AMAP_API_KEY": os.getenv("AMAP_API_KEY")},
                encoding="utf-8",
                encoding_error_handler="replace"
            ),
            timeout=10.0
        )
    )

    # 通过 tools 参数将 MCP 服务器接入到 Agent 中
    Agent = LlmAgent(
        name="agent",
        model=model,
        tools=[amap_mcp_server],
    )
```

&emsp;&emsp;这个其实不难理解。`stdio`既然是通过在本地服务器启动子进程的通信方式提供服务，所以我们是需要自主研发或者下载公开的`MCP`服务器至本地开发环境后，才能够通过`studio`的`MCPToolset`来实现与`MCP`服务器的交互。这里我们就尝试用高德地图的`MCP`服务来实现景点规划的功能。高德`MCP`可以提供地址与坐标转换、景点搜索、路线规划、距离计算、天气查询等工具，可以在如下`github`仓库中找到：https://github.com/sugarforever/amap-mcp-server 。


<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507271430897.png" alt="MCPToolset" style="width: 80%;">
</div>


- **Step 1. 下载源码文件并使用IDE打开** 

&emsp;&emsp;&emsp;&emsp;首先，我们要把高德地图的`MCP`服务运行起来，即先把`amap-mcp-server`仓库的代码下载值本地。大家可以直接在[github](https://github.com/sugarforever/amap-mcp-server)下载，也可以在课件的百度网盘中直接下载使用。我们这里使用`Cursor`进行项目的开发。注意：建议大家使用我们网盘中的课件版本，因为在下面的课程中我们会对公开的`amap-mcp-server`进行功能的扩展。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507271500295.png" alt="MCPToolset" style="width: 80%;">
</div>

- **Step 2. 注册高德地图开发者账号**

&emsp;&emsp;`amap-mcp-server`的实现功能都是基于高德的`API`，因此我们首先需要获取高德的`API`密钥。高德的`API`密钥可以在[高德开放平台](https://lbs.amap.com/dev/key/app)中获取。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507271430889.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;注册登录后，点击控制台进入：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507271430891.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;这里可以看到当前账户下的`Key`，如果是新用户，点击管理`Key`:

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507271430892.png" alt="MCPToolset" style="width: 80%;">
</div>


&emsp;&emsp;进入新页面后，点击创建新应用，输入应用名称，点击创建。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507271430893.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;创建完成后，选择该应用，并点击添加`Key`。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507271430894.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;输入`Key`名称，并选择为`Web服务`类型，点击提交按钮即可。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507271430895.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;至此，高德地图的`API`密钥就获取到了。接下来，我们就可以在`amap-mcp-server`的`.env`文件中配置`API`密钥。

- **Step 3. 配置`API`密钥**

&emsp;&emsp;在我们提供的`amap-mcp-server`仓库中有一个`.env`文件，通过`AMAP_MAPS_API_KEY`配置上一步获取的高德地图的`API`密钥。如下所示：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507281100936.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;配置完成后，我们就可以在`Google ADK`框架中接入并运行`amap-mcp-server`了。

- **Step 4. ADK接入MCP服务**

&emsp;&emsp;在通过`Google ADK`框架接入`MCP`服务时，我们只需要按照`StdioConnectionParams`的参数配置`MCP`服务器的连接参数即可。而对于`stdio`类型的`MCP`服务器，最关键的核心就是找到`MCP`服务器的`server.py`文件，并将其路径配置到`StdioConnectionParams`的`command`参数中。这里推荐的写法如下：

```python
    # stdio_agent/agent.py的20和22行，获取当前文件的绝对路径
    current_dir = os.path.dirname(os.path.abspath(__file__))
    # 提取 MCP 服务器路径
    server_path = os.path.join(current_dir, "..", "..", "mcp_servers", "amap-mcp-server", "amap_mcp_server", "server.py")
```

&emsp;&emsp;其中`current_dir`指的是当前`agent.py`的路径，每一个`..`，指代相对于当前目录的上一级目录。因此，`server_path`就是`amap-mcp-server`仓库的`server.py`文件的路径。大家需要根据自己实际的文件结构进行调整。

&emsp;&emsp;完整的代码在`adk_mcp_agent`文件夹中的`stdio_agent`文件夹中的`agent.py`文件中。启动方式如下所示：

```bash
    adk web
```

&emsp;&emsp;启动后，可以访问`http://localhost:8000`，进入`Google ADK`的`Web`界面。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291643705.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;在`Web`界面中，选择`stdio_agent`，并进行测试。效果如下所示：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291643706.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;只要能够正常的调用到某个`MCP Server`中的工具，则说明`Google ADK`框架已经成功接入`MCP`服务。整个接入过程其实并不复杂。

## 2.2 Http+Sse通信模式 

&emsp;&emsp;`Http + Sse`通信模式下，服务器端会独立运行（`amap-mcp-server`），而客户端（`ADK Agent`）则需要使用`HTTP POST`发送请求，并通过 `SSE (HTTP GET)` 建立长连接接收服务器（`amap-mcp-server`）推送的消息。

&emsp;&emsp;在`Google ADK`框架下，`Http + Sse` 的通信方式借助于`SseConnectionParams`来实现，其源码定义如下：


```python
    # adk-python-main\src\google\adk\tools\mcp_tool\mcp_session_manager.py
    elif isinstance(self._connection_params, SseConnectionParams):
      client = sse_client(
          url=self._connection_params.url,
          headers=merged_headers,
          timeout=self._connection_params.timeout,
          sse_read_timeout=self._connection_params.sse_read_timeout,
      )
```

&emsp;&emsp;`SseConnectionParams` 是`MCPSessionManager`的属性，用于存储`Sse`类型的`MCP`服务器连接参数。其核心定义如下：

```python

    # adk-python-main\src\google\adk\tools\mcp_tool\mcp_session_manager.py
    class SseConnectionParams(BaseModel):
      """Parameters for the MCP SSE connection.

      See MCP SSE Client documentation for more details.
      https://github.com/modelcontextprotocol/python-sdk/blob/main/src/mcp/client/sse.py

      Attributes:
          url: URL for the MCP SSE server.
          headers: Headers for the MCP SSE connection.
          timeout: Timeout in seconds for establishing the connection to the MCP SSE
            server.
          sse_read_timeout: Timeout in seconds for reading data from the MCP SSE
            server.
      """

        url: str
        headers: dict[str, Any] | None = None
        timeout: float = 5.0
        sse_read_timeout: float = 60 * 5.0
```

&emsp;&emsp;`SseConnectionParams` 一共只有四个参数属性，如下所示：

<style>
.center {
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size="4">StdioServerParameters 参数</font></p>
<div class="center">

| 参数 | 类型 / 默认值 | 说明 | 常见用法示例 |
| --- | --- | --- | --- |
| `url` | `str` **(必填)** | MCP SSE 服务器的 URL 地址 | `"http://localhost:8000/sse"` |
| `headers` | `dict[str, Any] \| None` | MCP SSE 连接的请求头 | `{"Authorization": "Bearer token"}` |
| `timeout` | `float = 5.0` | 建立连接到 MCP SSE 服务器的超时时间（秒） | `10.0` |
| `sse_read_timeout` | `float = 300.0` | 从 MCP SSE 服务器读取数据的超时时间（秒） | `600.0` |
</div>


&emsp;&emsp;对于支持`Http + Sse`通信模式的`MCP`服务器，我们需要先在本地服务器是启动`MCP`服务，然后通过`SseConnectionParams`来连接到`MCP`服务。比如我们现在使用的`amap-mcp-server`，其启动方式如下所示：

```bash
    cd mcp_servers/amap-mcp-server
    python server.py sse
```

&emsp;&emsp;运行效果如下图所示：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291709725.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;启动后，我们就可以通过`SseConnectionParams`来连接到`MCP`服务。具体的应用代码在`adk_mcp_agent`文件夹中的`sse_agent`文件夹中的`agent.py`文件中。同时需要注意的是：因为`amap-mcp-server`的服务器端口也是`8000`, 而`adk web` 的默认端口也是`8000`,所以可以通过`--port` 参数指定`adk web`的端口。如下所示:

```bash
    adk web --port 3000
```

&emsp;&emsp;启动后，我们就可以通过`http://localhost:3000` 访问`adk web`的界面。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291719034.png" alt="MCPToolset" style="width: 80%;">
</div>


&emsp;&emsp;在`adk web`的界面中，选择`sse_agent`，并进行测试。效果如下图所示：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291719035.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;同时，也可以在`amap-mcp-server`服务终端看到实时的请求信息。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291719036.png" alt="MCPToolset" style="width: 80%;">
</div>


&emsp;&emsp;这个接入过程也并不复杂，大家可以在不同的机器上部署`MCP`服务器，只要是网络连通，就可以通过`SseConnectionParams`来连接到`MCP`服务。但需要注意的是：一定要保持`MCP Server` 处于运行状态。

## 2.3 StreamableHTTP通信模式

&emsp;&emsp;`Streamable HTTP` 是 `MCP （Model Context Protocol）`在 `2025-03-26` 版规范中新引入的远程传输机制。该通信模式下可以用「同一个 HTTP 端点同时支持 POST + GET」取代了旧的「POST + SSE 双端点」模式，使客户端和服务器可以在一次 `POST` 里把 `JSON-RPC` 请求发过去，服务器必要时再把连接“升级”为 `SSE` 流，把多条响应、进度或反向请求实时推送回来。这让远程 `MCP` 服务器不必保持长连接，天然适合无状态 / Serverless 部署，也能与旧 SSE 模式共存。是目前效率最高，且最适用于企业环境下的通信模式。


&emsp;&emsp;在`Google ADK`框架下，`StreamableHTTP` 的通信方式借助于`StreamableHTTPConnectionParams`来实现，其源码定义如下：

```python
    # adk-python-main\src\google\adk\tools\mcp_tool\mcp_session_manager.py
    elif isinstance(self._connection_params, StreamableHTTPConnectionParams):
      client = streamablehttp_client(
          url=self._connection_params.url,
          headers=merged_headers,
          timeout=timedelta(seconds=self._connection_params.timeout),
          sse_read_timeout=timedelta(
              seconds=self._connection_params.sse_read_timeout
          ),
          terminate_on_close=self._connection_params.terminate_on_close,
      )
```

&emsp;&emsp;`StreamableHTTPConnectionParams` 是`MCPSessionManager`的属性，用于存储`StreamableHTTP`类型的`MCP`服务器连接参数。其核心定义如下：

```python

    # adk-python-main\src\google\adk\tools\mcp_tool\mcp_session_manager.py
    class StreamableHTTPConnectionParams(BaseModel):
    """Parameters for the MCP SSE connection.

    See MCP SSE Client documentation for more details.
    https://github.com/modelcontextprotocol/python-sdk/blob/main/src/mcp/client/streamable_http.py

    Attributes:
        url: URL for the MCP Streamable HTTP server.
        headers: Headers for the MCP Streamable HTTP connection.
        timeout: Timeout in seconds for establishing the connection to the MCP
            Streamable HTTP server.
        sse_read_timeout: Timeout in seconds for reading data from the MCP
            Streamable HTTP server.
        terminate_on_close: Whether to terminate the MCP Streamable HTTP server
            when the connection is closed.
    """

        url: str
        headers: dict[str, Any] | None = None
        timeout: float = 5.0
        sse_read_timeout: float = 60 * 5.0
        terminate_on_close: bool = True
```

&emsp;&emsp;`StreamableHTTPConnectionParams` 一共只有五个参数属性，如下所示：

<style>
.center {
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size="4">StreamableHTTPConnectionParams 参数</font></p>
<div class="center">

| 参数 | 类型 / 默认值 | 说明 | 常见用法示例 |
| --- | --- | --- | --- |
| `url` | `str` **(必填)** | MCP SSE 服务器的 URL 地址 | `"http://localhost:8000/mcp"` |
| `headers` | `dict[str, Any] \| None` | MCP SSE 连接的请求头 | `{"Authorization": "Bearer token"}` |
| `timeout` | `float = 5.0` | 建立连接到 MCP SSE 服务器的超时时间（秒） | `10.0` |
| `sse_read_timeout` | `float = 300.0` | 从 MCP SSE 服务器读取数据的超时时间（秒） | `600.0` |
| `terminate_on_close` | `bool = True` | 当连接关闭时是否终止 MCP 会话 | `True` |
</div>

&emsp;&emsp;`StreamableHTTPConnectionParams` 用于告诉 `ADK` 如何连接 `MCP Streamable HTTP` 服务器（单端点同时处理请求与 `SSE` 流）。参数与旧的 `SseServerParameters` 基本一致，但多了 `terminate_on_close`，这是 `Streamable HTTP` 独有的参数，控制当客户端断开连接时是否要求服务器端终止，以配合新版传输层的会话管理。具体协议细节可以参考：https://modelcontextprotocol.io/specification/2025-03-26/basic/transports ，我们这里不展开说明。

&emsp;&emsp;目前已经有越来越多的公开`MCP Server` 均支持`Streamable HTTP`通信模式，对于这类通信模式下的`MCP Server`其本质与`Http + Sse`通信模式下的`MCP Server`使用方法一样，我们依然需要先在本地服务器是启动`MCP`服务，然后通过`StreamableHTTPConnectionParams`来连接到`MCP`服务。

```bash
    cd mcp_servers/amap-mcp-server
    python server.py streamable-http
```

&emsp;&emsp;运行效果如下图所示：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291753538.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;启动后，我们就可以通过`StreamableHTTPConnectionParams`来连接到`MCP`服务。具体的应用代码在`adk_mcp_agent`文件夹中的`streamablehttp_agent`文件夹中的`agent.py`文件中。同时需要注意的是：因为`amap-mcp-server`的服务器端口也是`8000`, 而`adk web` 的默认端口也是`8000`,所以可以通过`--port` 参数指定`adk web`的端口。如下所示:

```bash
    adk web --port 3000
```

&emsp;&emsp;启动后，我们就可以通过`http://localhost:3000` 访问`adk web`的界面。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291719034.png" alt="MCPToolset" style="width: 80%;">
</div>


&emsp;&emsp;在`adk web`的界面中，选择`streamablehttp_agent`，并进行测试。效果如下图所示：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291753540.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;同时，也可以在`amap-mcp-server`服务终端看到实时的请求信息。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291753539.png" alt="MCPToolset" style="width: 80%;">
</div>


&emsp;&emsp;以上就是接入`Streamable HTTP`通信模式下的`MCP Server`的全部流程。对于多个`MCP Server`的接入也是一样，只需要在`tools`参数列表中添加多个`MCPToolset`即可。大家可以自行尝试。当然我们在课程后面的案例中也会涉及更多`Multi-MCP`的接入实战

&emsp;&emsp;对于`Agent` 接入`MCP` 的性能瓶颈，主要来源于传输协议的选择、工具元数据的体积、模型上下文膨胀及服务器启动与复用策略几个方面。如果采用`stido`模式，每次随代理进程一起启动，就会在首次调用时阻塞 0~N秒，复杂服务时间更长，这个冷启动的开销是很大的。此外，采用`HTTP + Sse`模式，虽然可以避免冷启动问题，但每次调用时都需要建立连接，一次`list_tools`就会把几十个工具、上百行的`Json Schema`都塞进上下文，直接加重`token`响应延时等多种情况。所以目前比较高效的就是`Streamable HTTP`模式 + 针对性的优化策略。

&emsp;&emsp;因此接下来，我们就详细介绍一些比较高效的`MCP` 接入策略，并重点介绍`ADK`接入`MCP`服务器的工具过滤及安全认证机制，这两个技术点在实际的应用开发中非常重要。

# 3. MCP 工具加载原理及过滤机制

&emsp;&emsp;在接入 `MCP（Model Context Protocol）`时，绝大多数代理框架都会先自动拉取每个 `MCP` 服务器的工具清单（`manifest`），再把其中每个函数的 `name / description / JSON Schema` 参数等元数据转成大模型能识别的 `tool definitions` 并随同大模型的调用一并发送。因此，函数描述都会被“加载”进模型上下文，让模型在推理时能够判断何时、如何调用这些外部函数。

&emsp;&emsp;`Google ADK` 框架的实现就完全遵循这种规范，其具体的执行过程如下所示：

```python
   # adk-python-main\src\google\adk\tools\mcp_tool\mcp_toolset.py
   # Step 1. 获取 MCP 服务器的所有工具，包括 name、description、inputSchema 等
   tools_response: ListToolsResult = await session.list_tools()

   # adk-python-main\src\google\adk\tools\mcp_tool\mcp_tool.py
   # Step 2. 将 MCP 服务器的工具转换为 ADK 的工具
    @override
    def _get_declaration(self) -> FunctionDeclaration:
        """Gets the function declaration for the tool."""
        schema_dict = self._mcp_tool.inputSchema
        parameters = _to_gemini_schema(schema_dict)   # 将 MCP 的 JSON Schema 转换为 Gemini 格式
        function_decl = FunctionDeclaration(  # 创建 FunctionDeclaration 包含 name、description、parameters
            name=self.name, description=self.description, parameters=parameters
        )
        return function_decl

   # adk-python-main\src\google\adk\tools\mcp_tool\mcp_toolset.py
   # Step 3. 工具封装
   tools = []
    for tool in tools_response.tools:
        mcp_tool = MCPTool(
            mcp_tool=tool,
            mcp_session_manager=self._mcp_session_manager,
            auth_scheme=self._auth_scheme,
            auth_credential=self._auth_credential,
        )

        if self._is_tool_selected(mcp_tool, readonly_context):
            tools.append(mcp_tool)
    return tools
```

&emsp;&emsp;这里我们可以快速的进行测试。
MCP服务器要先启动。在amap-mcp-server/amap_mcp_server下运行：python server.py streamable-http。

在`adk_mcp_agent`文件夹的`test_tools_list.py`文件中，20行url端口修改为8001.运行方式为：

```bash
    python test_tools_list.py
```

&emsp;&emsp;运行后，我们可以看到`amap-mcp-server`的工具清单列表。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291825992.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;因此，在使用外部`MCP`服务时，只加载真正需要的`MCP Tools`可以有效避免无意义的`Token`消耗。这一功能的实现在`Google ADK`框架中也有体现，我们只需要在使用`MCPToolset`时，在`tool_filter`参数中指定需要加载的`MCP Tools`即可。这里大家可以看`adk_mcp_agent`文件夹中的`stdio_agent_with_filter`文件夹中的`agent.py`文件，运行效果如下所示：

```bash
    adk web
```
<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291850857.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;运行后，我们可以看到`amap-mcp-server`的工具清单列表仅仅会包含在`tool_filter`参数中指定的`MCP Tools`。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507291849822.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;`tool_filter` 功能虽然在静态配置上看起来比较简单，但结合着`ADK` 的运行机制可以处理非常多复杂的`MCP` 接入场景。比如：

1. 如果`MCP Server` 下的工具很多，且比较固定，可以借助`Memory`来缓存`MCP Tools`的元数据，具体的思路是：将 `memory_service.add_session_to_memory()` 用于把一次 `list_tools()` 结果写入 `Memory`，下一轮用户查询时先检索记忆中最相关的工具集合，再动态注入到`tool_filter` 里，实现**多→少**的缩减;
2. 自定义 `ToolContext` 回调，根据用户角色、对话阶段或意图分支实时决定是否暴露工具，可以在降低`Token` 消耗的同时，实现灵活的用户、会话权限控制；
3. `Multi-Agent`架构下，用子 `Agent` 封装每个 `MCP Server`，再由根 `Agent` 规划之间的协同工作。

   ......
   ......

&emsp;&emsp;具体的策略要结合实际的开发需求和场景选择。当然，我们在最后一个项目`Manus`通用智能体项目中也会重点进行应用。

# 4. MCP 工具认证组件

&emsp;&emsp;`ADK` 提供了一套统一机制来处理外部工具(`API`)的身份认证。在实际业务开发中，这意味着我们开发者可以方便地将第三方服务功能集成到`AI Agent`中，同时确保调用这些服务时符合安全要求。比如对于一些需要用户授权的数据访问（如登录事件），`ADK`就是用内置的认证组件实现交互式授权流程的支持。也就是说，当`ADK Agent`尝试访问、使用需要授权的工具时，若缺少有效凭证，`ADK`会暂停`Agent`执行并触发一个特殊事件请求用户完成登录/授权。开发者只需按`ADK`流程要求在客户端重定向用户进行授权、回调后将授权码交回，`ADK`便会自动换取访问令牌供`Agent`使用。

&emsp;&emsp;简单理解就是：在`ADK`里，任何需要凭证才能调用的工具（不论是最简单的 `API Key` 还是完整的 `OAuth2` 交互流程）都会通过同一套`AuthScheme + AuthCredential + 框架事件流`机制嵌入到 `Agent` 的运行链路。

&emsp;&emsp;在上面的案例中，在启动后大家应该也都注意到了如下的警告信息：

```bash
    auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not require
```

&emsp;&emsp;通过源码可以找到源头是来自 `base_authenticated_tool.py` 文件，即如下：

```python
    # adk-python-main\src\google\adk\tools\base_authenticated_tool.py
    if auth_config and auth_config.auth_scheme:
      self._credentials_manager = CredentialManager(auth_config=auth_config)
    else:
      logger.warning(
          "auth_config or auth_config.auth_scheme is missing. Will skip"
          " authentication.Using FunctionTool instead if authentication is not"
          " required."
      )
      self._credentials_manager = None
    self._response_for_auth_required = response_for_auth_required
```

&emsp;&emsp;其中我们所使用的`MCPTool`继承自`BaseAuthenticatedTool`，源码如下：

```python
    # adk-python-main\src\google\adk\tools\mcp_tool\mcp_tool.py
    class MCPTool(BaseAuthenticatedTool):
        ....
        ....
        @retry_on_closed_resource
        async def get_tools(
            self,
            readonly_context: Optional[ReadonlyContext] = None,
        ) -> List[BaseTool]:
            # Step 1. 在 MCPToolset.get_tools() 中，传递认证配置
            mcp_tool = MCPTool(
                mcp_tool=tool,
                mcp_session_manager=self._mcp_session_manager,
                auth_scheme=self._auth_scheme,        # 从 toolset 传递
                auth_credential=self._auth_credential, # 从 toolset 传递
            )


    # adk-python-main\src\google\adk\tools\base_authenticated_tool.py
    class BaseAuthenticatedTool(BaseTool):
        super().__init__(
            name=name,
            description=description,
        )

        # Step 2.只有 auth_scheme 存在时才创建 AuthConfig
        if auth_config and auth_config.auth_scheme:
            self._credentials_manager = CredentialManager(auth_config=auth_config)
        else:
            logger.warning(
                "auth_config or auth_config.auth_scheme is missing. Will skip"
                " authentication.Using FunctionTool instead if authentication is not"
                " required."
            )
        self._credentials_manager = None
        self._response_for_auth_required = response_for_auth_required
```

&emsp;&emsp;从源码中能够非常清晰的分析出处理逻辑，就是因为我们的`MCP`工具没有设置 `auth_scheme` 和 `auth_credential`，所以才会发出`auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not required.`的警告，但并不影响使用，系统会将`auth_config`设置为`None`，作为默认值继续生成`FunctionTool`。

&emsp;&emsp;而如果要了解工具认证功能的使用场景，我们则需要了解`auth_scheme`和`auth_credential`的配置方式。首先来看`auth_scheme`。其源码定义：

```python   
    # adk-python-main\src\google\adk\auth\auth_tool.py
    class AuthConfig(BaseModelWithConfig):
    """The auth config sent by tool asking client to collect auth credentials and
        adk and client will help to fill in the response
    """

        auth_scheme: AuthScheme
```

&emsp;&emsp;其中：

```python
    AuthScheme = Union[SecurityScheme, OpenIdConnectWithConfig]
    # adk-python-main\src\google\adk\auth\auth_schemes.py
    SecurityScheme = Union[APIKey, HTTPBase, OAuth2, OpenIdConnect, HTTPBearer]
```

&emsp;&emsp;`AuthSchema`是用来描述认证的方案，即告诉`ADK`使用什么方式来认证。其源码定义如下：

```python
    # adk_venv\Lib\site-packages\google\adk\tools\openapi_tool\auth\auth_helpers.py
    # 支持 API Key 、OAuth2 Token 和 HTTP 三种认证方式，其中`oauth2Token`是Http Bearer 
    def token_to_scheme_credential(
        token_type: Literal["apikey", "oauth2Token"],
        location: Optional[Literal["header", "query", "cookie"]] = None,
        name: Optional[str] = None,
        credential_value: Optional[str] = None,
    ) -> Tuple[AuthScheme, AuthCredential]:


    # 支持 Service Account 认证方式
    def service_account_dict_to_scheme_credential(
        config: Dict[str, Any],
        scopes: List[str],
    ) -> Tuple[AuthScheme, AuthCredential]:

    # 支持 OpenID Connect 认证方式
    def openid_dict_to_scheme_credential(
        config_dict: Dict[str, Any],
        scopes: List[str],
        credential_dict: Dict[str, Any],
    ) -> Tuple[OpenIdConnectWithConfig, AuthCredential]:
```

&emsp;&emsp;如上三个定义包含五种认证方式：

<style>
.center {
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size="4"> ADK 内置认证方式</font></p>
<div class="center">

| 方式                            |  典型场景                                 | 交互/刷新特点                                                                                                                           |
| ----------------------------- | ------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------- |
| **API\_KEY**                  |  公共或统一 Key 的地图、天气、金融行情 API          | 无刷新，缺失时一次性输入即可                                                                         |
| **HTTP**<br/>（Basic / Bearer） |  私有 REST 服务、JWT Bearer Token          | Authorization: Bearer <token> 或 X-API-Key: <key>                                            |
| **OAUTH2**                    | 访问用户私有数据（Google Cloud） | 需浏览器重定向 → 回填 `auth_response_uri` → ADK 自动换 `access/refresh_token`                     |
| **SERVICE\_ACCOUNT**          | 服务器到服务器：调用 Vertex AI、BigQuery、GCS 等  | ADK 读取 JSON Key 或使用实例元数据|
| **OPEN\_ID\_CONNECT**         |  企业 SSO、OIDC‑compatible IdP           | 流程与 OAuth2 类似，但最终下发 ID Token；ADK 支持把 ID Token 写进 `Authorization: Bearer`                                |

&emsp;&emsp;这里要理解`refresh`在认证机制中的作用。刷新存在的根本原因是访问令牌有效期很短（一般 1 小时），短期令牌能减小泄露风险，但也要求客户端在过期后无感继续工作。`OAuth 2.0` 和 `OIDC` 规范因此引入 `refresh token`，本身寿命更长，可反复换得新的访问令牌。在 `ADK` 的认证体系里，“刷新（`refresh`）”指的是当现有访问凭据（`access token`）即将过期或已经失效时，自动或半自动地向身份提供方请求一组新的凭据，从而让工具调用不中断、用户不必重新登录。一般开发场景下会借助 `refresh token`、服务账号私钥或实例元数据来完成，`ADK`则是通过内置或自定义的 `CredentialRefresher` 机制，把这件事封装到后台。以`OAuth2`为例，其认证流程如下：

1. **首次鉴权**：用户通过浏览器完成授权码流程，`ADK`在 `adk_request_credential` 回调中用授权码换得 `(access_token, refresh_token)` 并写入 `session.state`。

2. **过期检测**：`ADK`的 `AuthPreprocessor` 在每次调用前检查 `access_token` 的 `expires_in`；若已过期，则触发刷新的 `CredentialRefresher`。

3. **自动刷新**：`Google SDK` 会自动在到期前 5 分钟，执行`CredentialRefresher` 向 IdP 的 `token_endpoint` 发送 `grant_type=refresh_token`，换得新一对 `token` 并更新会话。

4. **重试原调用**：刷新成功后，`ADK` 自动重播之前失败的工具请求；用户侧无感知。

&emsp;&emsp;若刷新失败，`ADK`会再次抛 `adk_request_credential`，要求用户重新授权。其过程如下所示：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507281450665.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;上图的时序图中，整条链路展示了 当 `LLM‑Agent` 在调用需要 `OAuth 凭据`的 `BigQuery Tool` 时，`ADK` 如何暂停执行、向前端请求用户授权、完成令牌交换，再自动重试原始调用 的全过程。如下所示：

1. 用户提出查询后，`Agent` 客户端把问题递交给 `ADK`。

2. `Agent` 解析用户意图，决定调用 `BQ Tool`。

3. `ADK` 检查会话：发现没有可用 `access_token` 或已过期 → 抛出特殊函数调用 `adk_request_credential`。

4. 前端收到事件：根据事件中的 `authScheme` & `authConfig` 生成 `OAuth` 授权 URL → 浏览器跳转到 Google 账户登录/同意页。

5. 用户授权 后，重定向回客户端的 `callback` `URI`，并携带 `code`。

6. 前端把回调 `URL`封装进 `FunctionResponse` 发送给 `ADK`，`ADK` 自动用授权码换取 `(access_token, refresh_token)` 并持久化到 `session.state`。

7. `ADK` 重试工具调用：带上新的 `Bearer` `token` 调用 `BQ API`，`LLM` 获得数据后回复用户。

8. 后续请求：若 `token` 将过期，内置 `OAuth2CredentialRefresher` 在到期前 5 分钟用 `refresh` `token` 换新，整个过程对用户无感知

&emsp;&emsp;在进行认证方案选型时，比如快速构建应用原型，或者`MCP` 是通过公共只读的 `API_KEY` 请求，那么就使用`API_KEY`模式，简单且不需要回调，如果涉及用户数据，则可以使用`OAUTH2` 或 `OPEN_ID_CONNECT`，需前端配合重定向实现。具体大家以实际开发的需求灵活调整，我们这里不再额外扩展。

&emsp;&emsp;了解了`auth_scheme`，我们再来看`auth_credential`。`AuthCredential`（认证凭据）是`ADK`中用于表示认证信息的数据类，它包含了认证类型、认证凭据等重要信息。

```python
  # adk-python-main\src\google\adk\auth\auth_credential.py
    class AuthCredential(BaseModelWithConfig):
      api_key: Optional[str] = None
      http: Optional[HttpAuth] = None
      service_account: Optional[ServiceAccount] = None
      oauth2: Optional[OAuth2Auth] = None
```

&emsp;&emsp;`AuthCredential`其实非常容易理解，它做的事情就是：遵循不同认证协议规范（`apikey`、`oauth2`、`service_account`）编写得到具体的实例类，接收`AuthScheme`传递过来的认证参数，将其进行转化成对应的认证信息。所以`AuthScheme`和`AuthCredential`的实际执行流程为：

```json
    配置阶段：
    AuthScheme (定义认证方式) + AuthCredential (提供初始凭据)
        ↓
    运行时处理：
    ADK 认证处理 (CredentialManager)
        ↓
    最终使用：
    按 AuthScheme 定义的格式使用最终认证信息
```

&emsp;&emsp;`AuthScheme` 定义格式，`AuthCredential` 提供内容，`ADK` 处理转换，最终按格式使用。因此，在实际使用的时候，我们只需要配置`AuthScheme`的认证方式，`AuthCredential`会自动处理所有细节，包括认证验证、凭据交换、格式转换、请求添加等。

&emsp;&emsp;应用方法如下所示，大家可以直接运行`test_mcp_auth.py`文件来测试。

```bash
    python test_mcp_auth.py
```

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507292008625.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;运行后，我们可以看到`ADK`的认证配置信息。这就是客户端的配置方式（`ADK Agent`），当`ADK Agent` 调用`MCP Server`时，会自动使用`AuthScheme` 和 `AuthCredential` 进行认证，那如果要想影响到工具的执行，自然我们也需要根据认证的请求类型针对性的修改`MCP Server`的认证处理逻辑。

&emsp;&emsp;理解了上述的原理后，我们接下来就实际的应用一下。但需要说明的是：`Google ADK`的认证组件目前还处于实验阶段，体系还不是很完善，同时在最新的版本中(`v1.8.0`)中使用`adk web`时会有`Bug`，因此我们需要在后台自行编写`Runner`来进行测试。因此，这里我们首先通过`test_adk_run.py`来跑通`ADK`的完整运行流程。

```bash
    python test_adk_run.py
```

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507292015784.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;运行后，我们可以看到`ADK`的完整运行流程。

&emsp;&emsp;接下来接入`amap-mcp-server`，并进行对话测试。这里我们使用`test_mcp_streamablehttp.py`运行。修改通讯端口为8001，先启动MCP服务器。

```bash
    python test_mcp_streamablehttp.py
```

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507292023020.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;运行后，我们可以看到现在已经可以正常的调用`amap-mcp-server`的工具了。接下来，我们在客户端(`ADK Agent`)添加认证组件。这里可以看`test_auth_apikey.py`文件，我们通过`token_to_scheme_credential`函数来创建认证组件，并将其传递给`MCPToolset`。运行效果如下所示：

```bash
    python test_auth_apikey.py
```

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507292112410.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;同时，在`amap-mcp-server`中添加需要执行身份认证的工具，如下所示：

```python
    @mcp.tool()
    def geocode(address: str, ctx: Context = None) -> dict:
        """
        将详细的结构化地址转换为经纬度坐标。支持对地标性名胜景区、建筑物名称解析为经纬度坐标

        Args:
            address (str): 详细的结构化地址（如"北京市朝阳区阜通东大街6号"）
            ctx (Context): 上下文对象，包含请求信息。
        Returns:
            {
                "status": "success" | "error",
                "password":       "<随机密码>",     # status=success 时存在
                "error_message":  "<错误信息>"      # status=error  时存在
            }
        """
        # 完整获取 ADK 传递的请求信息
        if ctx and hasattr(ctx, 'request_context') and ctx.request_context:
            try:
                request = ctx.request_context.request
                print("=== ADK 请求信息 ===")
                print(f"请求方法: {request.method}")
                print(f"请求 URL: {request.url}")
                print(f"请求路径: {request.url.path}")
                print(f"请求查询参数: {request.url.query}")
                
                # 获取所有 headers
                print("=== Headers ===")
                for name, value in request.headers.items():
                    print(f"  {name}: {value}")
            except Exception as e:
                print(f"无法获取完整请求信息: {e}")
        
        # 根据传递的 API 调用产生调用
        # TODO ：按照这种思路，就可以根据传递进来的值，结合具体的场景实现路由、权限控制等多种情况
        try:
            response = requests.get(
                "https://restapi.amap.com/v3/geocode/geo",
                params={"address": address, "key": request.headers.get("X-API-Key")},
                timeout=5,
            )
            
            # 实际获取数据
            response.raise_for_status()
            data = response.json()
            
            if data["status"] != "1":
                return {
                    "status": "error",
                    "message": "调用异常，该用户无权限",
                    "error_message": f"Geocoding failed: {data.get('info') or data.get('infocode')}"
                }
            
            # 如果请求成功，返回经纬度坐标
            geocodes = data.get("geocodes", [])
            results = []
            for geo in geocodes:
                results.append({
                    "location": geo.get("location"),
                    "level": geo.get("level")
                })
            return {
                "status": "success",
                "message": "调用成功，返回经纬度坐标",
                "return": results,
            }
        except requests.exceptions.RequestException as e:
            return {
                "status": "error",
                "error_message": f"Request failed: {str(e)}"
            }
```

&emsp;&emsp;当触发此工具的请求时，会接收`AuthScheme`和`AuthCredential`，并根据传递的值进行认证。认证成功后，才会实际执行`geocode`的工具。运行效果如下所示：

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507292115118.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;而如果传入的用户信息无法通过认证，则会返回错误信息。

<div style="text-align: center;">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202507292117351.png" alt="MCPToolset" style="width: 80%;">
</div>

&emsp;&emsp;而这样的方式，仅仅会影响具体需要认证信息的工具，完全不会影响其他工具的正常执行，从而实现更加灵活的工具执行权限、路由控制等。

&emsp;&emsp;大家在实际进行开发时，只要`Agent`涉及“身份”或“权限”的`API`调用场景，都建议使用该功能，包括但不限于用户登录授权、服务身份认证、隐私数据访问、跨系统操作等。这既保障系统安全，又提升开发效率，因为`ADK`已为这些复杂流程提供了开箱即用的支持。但需要说明的是，该认证组件不是控制用户访问工具，而是确保工具能成功调用外部 `API`，只不过通过合理的流程控制，我们可以再次基础上非常方便的实现更加复杂的权限控制。外层通过`tool_filter`控制可以使用的工具，内层通过`auth_scheme`和`auth_credential`组件来控制更细粒度的工具执行权限，这种形式是我们目前在基于`Google ADK`框架进行开发中探索出来的比较高效且灵活的开发范式，大家可以在此基础上进一步探索和优化。